# Lateral-Torsional Buckling — Shell Model (ASDShellQ4)

This notebook demonstrates **lateral-torsional buckling** of a simply-supported
I-beam under uniform moment, modelled with **ASDShellQ4** shell elements in
corotational formulation.

**Key features demonstrated:**
- Manual I-beam mid-surface geometry (3 plates: web + flanges)
- Transfinite quad mesh
- Mesh-level half-sine imperfection (node perturbation)
- `face_load` for applying end moments directly to edge nodes
- `rigidLink` reference nodes for fork-support BCs
- Native openseespy with ASDShellQ4 `-corotational`
- Comparison to classical $M_{cr} = \frac{\pi}{L}\sqrt{EI_w \cdot GJ}$

### Classical LTB critical moment

For a doubly-symmetric I-section under uniform moment with fork supports:

$$M_{cr} = \frac{\pi}{L}\sqrt{E I_w \cdot G J}$$

where $I_w$ is the weak-axis second moment of area and $J$ is the
St. Venant torsion constant.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Section properties (mm, N, MPa) ─────────────────────
bf   = 127.0       # flange width
tf   = 8.5         # flange thickness
h    = 332.0       # clear web height (between flange mid-surfaces)
tw   = 5.8         # web thickness
L    = 4000.0      # span
d    = h + tf       # total depth to flange mid-surfaces

E    = 200_000.0   # Young's modulus (MPa)
nu   = 0.3
G    = E / (2 * (1 + nu))

# Derived section properties
I_weak = (2 * tf * bf**3 + h * tw**3) / 12   # weak-axis I
J      = (2 * bf * tf**3 + h * tw**3) / 3     # torsion constant

M_cr = (np.pi / L) * np.sqrt(E * I_weak * G * J)
print(f"I_weak = {I_weak:.0f} mm^4")
print(f"J      = {J:.0f} mm^4")
print(f"M_cr   = {M_cr/1e6:.3f} kN·m")

## 1. Geometry — Three mid-surface plates

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../src'))

from apeGmsh import apeGmsh
import gmsh

g = apeGmsh(model_name='LTB_shell')
g.begin()
geo = g.model.geometry

y_tf = d / 2      # top flange mid-surface elevation
y_bf = -d / 2     # bottom flange mid-surface elevation

def add_plate(x0, y0, z0, x1, y1, z1, x2, y2, z2, x3, y3, z3, label):
    """Create a planar quad surface from 4 corners."""
    p1 = geo.add_point(x0, y0, z0, sync=False)
    p2 = geo.add_point(x1, y1, z1, sync=False)
    p3 = geo.add_point(x2, y2, z2, sync=False)
    p4 = geo.add_point(x3, y3, z3, sync=False)
    l1 = geo.add_line(p1, p2, sync=False)
    l2 = geo.add_line(p2, p3, sync=False)
    l3 = geo.add_line(p3, p4, sync=False)
    l4 = geo.add_line(p4, p1, sync=False)
    cl = geo.add_curve_loop([l1, l2, l3, l4], sync=False)
    s  = geo.add_plane_surface(cl, sync=True, label=label)
    return s, [p1, p2, p3, p4], [l1, l2, l3, l4]

# Top flange (XZ plane at y = y_tf)
s_tf, pts_tf, lns_tf = add_plate(
    -bf/2, y_tf, 0,   bf/2, y_tf, 0,
     bf/2, y_tf, L,  -bf/2, y_tf, L,
    label='top_flange')

# Bottom flange (XZ plane at y = y_bf)
s_bf, pts_bf, lns_bf = add_plate(
    -bf/2, y_bf, 0,   bf/2, y_bf, 0,
     bf/2, y_bf, L,  -bf/2, y_bf, L,
    label='bot_flange')

# Web (YZ plane at x = 0)
s_web, pts_web, lns_web = add_plate(
    0, -h/2, 0,   0, h/2, 0,
    0,  h/2, L,   0, -h/2, L,
    label='web')

print(f"Surfaces: top_flange={s_tf}, bot_flange={s_bf}, web={s_web}")

## 2. Physical groups for BCs

In [ ]:
# End edges at z=0 and z=L for each plate
# From the add_plate layout: l1=cross@z=0, l2=long_right, l3=cross@z=L, l4=long_left

edges_z0 = [lns_tf[0], lns_bf[0], lns_web[0]]
edges_zL = [lns_tf[2], lns_bf[2], lns_web[2]]

# Create physical groups for the end cross-section edges
g.physical.add_curve(edges_z0, name='left_end')
g.physical.add_curve(edges_zL, name='right_end')

print(f"left_end edges:  {edges_z0}")
print(f"right_end edges: {edges_zL}")

## 3. Transfinite mesh

In [ ]:
# Mesh control: element count along each edge
n_span   = 40   # elements along the span (z-direction)
n_flange = 6    # elements across flange width (x-direction)
n_web    = 16   # elements along web height (y-direction)

mesh_struct = g.mesh.structured

# Top flange
mesh_struct.set_transfinite_curve(lns_tf[0], n_flange + 1)  # cross z=0
mesh_struct.set_transfinite_curve(lns_tf[2], n_flange + 1)  # cross z=L
mesh_struct.set_transfinite_curve(lns_tf[1], n_span + 1)    # long right
mesh_struct.set_transfinite_curve(lns_tf[3], n_span + 1)    # long left
mesh_struct.set_transfinite_surface(s_tf, arrangement='Left')

# Bottom flange
mesh_struct.set_transfinite_curve(lns_bf[0], n_flange + 1)
mesh_struct.set_transfinite_curve(lns_bf[2], n_flange + 1)
mesh_struct.set_transfinite_curve(lns_bf[1], n_span + 1)
mesh_struct.set_transfinite_curve(lns_bf[3], n_span + 1)
mesh_struct.set_transfinite_surface(s_bf, arrangement='Left')

# Web
mesh_struct.set_transfinite_curve(lns_web[0], n_web + 1)
mesh_struct.set_transfinite_curve(lns_web[2], n_web + 1)
mesh_struct.set_transfinite_curve(lns_web[1], n_span + 1)
mesh_struct.set_transfinite_curve(lns_web[3], n_span + 1)
mesh_struct.set_transfinite_surface(s_web, arrangement='Left')

# Recombine for quads
gmsh.model.mesh.setRecombine(2, s_tf)
gmsh.model.mesh.setRecombine(2, s_bf)
gmsh.model.mesh.setRecombine(2, s_web)

# Generate
g.mesh.generation.generate(dim=2)

# Stats
info = gmsh.model.mesh.getNodes()
print(f"Nodes: {len(info[0])}")
for s_tag in [s_tf, s_bf, s_web]:
    etypes, etags, _ = gmsh.model.mesh.getElements(2, s_tag)
    n_elem = sum(len(t) for t in etags)
    print(f"  Surface {s_tag}: {n_elem} elements (type {list(etypes)})")

## 4. Apply half-sine imperfection to mesh nodes

Perturb all mesh nodes laterally (x-direction) by a half-sine
envelope along the span: $\Delta x = \delta_0 \sin(\pi z / L)$
where $\delta_0 = L / 1000$.

In [ ]:
imp_mag = L / 1000  # 4 mm imperfection at midspan

all_tags, all_coords, _ = gmsh.model.mesh.getNodes()
coords = np.array(all_coords).reshape(-1, 3)

# Half-sine perturbation in x, keyed by z
z_vals = coords[:, 2]
dx = imp_mag * np.sin(np.pi * z_vals / L)

coords[:, 0] += dx

# Write back perturbed coordinates
for i, tag in enumerate(all_tags):
    gmsh.model.mesh.setNode(int(tag), list(coords[i]), [])

print(f"Max lateral perturbation: {dx.max():.2f} mm at midspan")
print(f"Imperfection ratio: L/{int(L/imp_mag)}")

## 5. FEM data extraction + face_load

In [ ]:
# ── Loads: equal-and-opposite strong-axis moments ─────────
with g.loads.pattern('moment'):
    g.loads.face_load(pg='left_end',  moment_xyz=(0, M_cr, 0))
    g.loads.face_load(pg='right_end', moment_xyz=(0, -M_cr, 0))

# ── BCs: face_sp for direct support (no rigidLink needed) ──
# Left end: pin — fix all 3 translations
g.loads.face_sp(pg='left_end', dofs=[1, 1, 1])
# Right end: roller — fix ux, uy (uz free for axial expansion)
g.loads.face_sp(pg='right_end', dofs=[1, 1, 0])

# Extract FEM data
fem = g.mesh.queries.get_fem_data(dim=2)

print(f"Nodes: {len(fem.nodes.ids)}")
print(f"Loads: {fem.nodes.loads}")
print(f"SP:    {fem.nodes.sp}")

# Verify total moment from face_load
id_to_idx = {int(nid): i for i, nid in enumerate(fem.nodes.ids)}
total_m = np.zeros(3)
for rec in fem.nodes.loads:
    if rec.force_xyz is not None:
        r = fem.nodes.coords[id_to_idx[rec.node_id]]
        total_m += np.cross(r, np.array(rec.force_xyz))
print(f"\nTotal moment about origin: {total_m / 1e6} (kN·m)")

## 6. OpenSees model — native openseespy

ASDShellQ4 with `-corotational` flag for geometric nonlinearity.
BCs applied directly to end nodes via `face_sp` (no reference node / rigidLink needed).
Moments applied as distributed force couples via `face_load`.

In [ ]:
import subprocess

venv_python = r'C:\Users\nmora\venv\opensees_venv\Scripts\python.exe'

# ── Node data ─────────────────────────────────────────────
node_data = []
for nid, xyz in fem.nodes.get():
    node_data.append((int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2])))

# ── Element data by label (for different shell thicknesses) ──
tf_ids, tf_conn = fem.elements.get(label='top_flange').resolve()
bf_ids, bf_conn = fem.elements.get(label='bot_flange').resolve()
web_ids, web_conn = fem.elements.get(label='web').resolve()

elem_tf  = [(int(tf_ids[i]),  [int(c) for c in tf_conn[i]])  for i in range(len(tf_ids))]
elem_bf  = [(int(bf_ids[i]),  [int(c) for c in bf_conn[i]])  for i in range(len(bf_ids))]
elem_web = [(int(web_ids[i]), [int(c) for c in web_conn[i]]) for i in range(len(web_ids))]

print(f"Elements: TF={len(elem_tf)}, BF={len(elem_bf)}, Web={len(elem_web)}")

# ── Load data ─────────────────────────────────────────────
load_data = []
for rec in fem.nodes.loads:
    f = rec.force_xyz or (0, 0, 0)
    m = rec.moment_xyz or (0, 0, 0)
    load_data.append((int(rec.node_id), f[0], f[1], f[2], m[0], m[1], m[2]))

# ── Edge nodes for rigidLink BCs ──────────────────────────
left_nodes  = sorted(fem.nodes.get_ids(pg='left_end'))
right_nodes = sorted(fem.nodes.get_ids(pg='right_end'))

# Reference node centroids
id_to_idx = {int(nid): i for i, nid in enumerate(fem.nodes.ids)}
left_coords  = np.array([fem.nodes.coords[id_to_idx[n]] for n in left_nodes])
right_coords = np.array([fem.nodes.coords[id_to_idx[n]] for n in right_nodes])
left_centroid  = left_coords.mean(axis=0)
right_centroid = right_coords.mean(axis=0)

# Midspan top-flange node for tracking lateral displacement
tf_node_ids = sorted(fem.nodes.get_ids(label='top_flange'))
tf_node_coords = np.array([fem.nodes.coords[id_to_idx[n]] for n in tf_node_ids])
target = np.array([0, y_tf, L/2])
dists = np.linalg.norm(tf_node_coords - target, axis=1)
midspan_node = tf_node_ids[np.argmin(dists)]

print(f"Tracking node: {midspan_node} at {fem.nodes.coords[id_to_idx[midspan_node]]}")
print(f"Left end: {len(left_nodes)} nodes, centroid={left_centroid}")
print(f"Right end: {len(right_nodes)} nodes, centroid={right_centroid}")

In [ ]:
# ── Build the openseespy script ──────────────────────────
n_steps = 80

# Node data
node_lines = []
for nid, x, y, z in node_data:
    node_lines.append(f"ops.node({nid}, {x:.6f}, {y:.6f}, {z:.6f})")

# Element lines
elem_lines = []
for i in range(len(tf_ids)):
    c = tf_conn[i]
    elem_lines.append(f"ops.element('ASDShellQ4', {int(tf_ids[i])}, {int(c[0])}, {int(c[1])}, {int(c[2])}, {int(c[3])}, 1, '-corotational')")
for i in range(len(bf_ids)):
    c = bf_conn[i]
    elem_lines.append(f"ops.element('ASDShellQ4', {int(bf_ids[i])}, {int(c[0])}, {int(c[1])}, {int(c[2])}, {int(c[3])}, 1, '-corotational')")
for i in range(len(web_ids)):
    c = web_conn[i]
    elem_lines.append(f"ops.element('ASDShellQ4', {int(web_ids[i])}, {int(c[0])}, {int(c[1])}, {int(c[2])}, {int(c[3])}, 2, '-corotational')")

# BCs from face_sp records (direct fix on end nodes — no rigidLink)
sp_fix = {}  # node_id -> set of dofs
for rec in fem.nodes.sp:
    sp_fix.setdefault(int(rec.node_id), set()).add(rec.dof)

fix_lines = []
for nid, dofs in sp_fix.items():
    mask = [1 if d in dofs else 0 for d in range(1, 7)]
    fix_lines.append(f"ops.fix({nid}, {mask[0]}, {mask[1]}, {mask[2]}, {mask[3]}, {mask[4]}, {mask[5]})")

# Load lines
load_lines = []
for rec in fem.nodes.loads:
    f = rec.force_xyz or (0, 0, 0)
    m = rec.moment_xyz or (0, 0, 0)
    load_lines.append(
        f"ops.load({int(rec.node_id)}, {f[0]:.10g}, {f[1]:.10g}, {f[2]:.10g}, "
        f"{m[0]:.10g}, {m[1]:.10g}, {m[2]:.10g})")

# Find midspan tracking nodes (top flange and bot flange)
bf_node_ids = sorted(int(n) for n in fem.nodes.get_ids(label='bot_flange'))
bf_node_coords = np.array([fem.nodes.coords[id_to_idx[n]] for n in bf_node_ids])
target_bf = np.array([0, y_bf, L/2])
midspan_bf = bf_node_ids[np.argmin(np.linalg.norm(bf_node_coords - target_bf, axis=1))]

print(f"Tracking: TF node {midspan_node}, BF node {midspan_bf}")

# Assemble script
script = f"""\
import openseespy.opensees as ops
import numpy as np
import json

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

# ── Nodes ─────────────────────────────────────────────────
{chr(10).join(node_lines)}

# ── Sections ──────────────────────────────────────────────
ops.section('ElasticMembranePlateSection', 1, {E}, {nu}, {tf}, 0.0)   # flanges
ops.section('ElasticMembranePlateSection', 2, {E}, {nu}, {tw}, 0.0)   # web

# ── Elements (ASDShellQ4 -corotational) ───────────────────
{chr(10).join(elem_lines)}

# ── Boundary conditions (face_sp: direct fix on end nodes) ──
{chr(10).join(fix_lines)}

# ── Loading (face_load: distributed moment couples) ───────
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)
{chr(10).join(load_lines)}

# ── Analysis ──────────────────────────────────────────────
ops.constraints('Plain')
ops.numberer('Plain')
ops.system('UmfPack')
ops.test('NormDispIncr', 1e-5, 100)
ops.algorithm('Newton')

n_steps = {n_steps}
dLambda = 1.5 / n_steps
ops.integrator('LoadControl', dLambda)
ops.analysis('Static')

lambdas = []
ux_tf = []   # top flange lateral displacement
ux_bf = []   # bottom flange lateral displacement

for step in range(n_steps):
    ok = ops.analyze(1)
    if ok != 0:
        print(f"Analysis failed at step {{step}}, lambda={{ops.getTime():.4f}}")
        break
    lam = ops.getTime()
    u_top = ops.nodeDisp({midspan_node}, 1)
    u_bot = ops.nodeDisp({midspan_bf}, 1)
    lambdas.append(lam)
    ux_tf.append(u_top)
    ux_bf.append(u_bot)
    if step % 10 == 0:
        twist = abs(u_top - u_bot)
        print(f"  step {{step:3d}}: lambda={{lam:.4f}}, ux_tf={{u_top:.3f}}, twist={{twist:.3f}}")

print(f"\\nCompleted {{len(lambdas)}} steps, final lambda = {{lambdas[-1]:.4f}}")

results = {{
    'lambdas': lambdas,
    'ux_tf': ux_tf,
    'ux_bf': ux_bf,
    'M_cr': {M_cr},
}}
with open('ltb_shell_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f)
print("Results saved")
"""

script_path = 'ltb_shell_analysis.py'
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(script)
print(f"Script: {len(script.splitlines())} lines, {len(elem_lines)} elements")

## 7. Run analysis

In [ ]:
result = subprocess.run(
    [venv_python, script_path],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## 8. Results & comparison to classical M_cr

In [ ]:
import json

with open('ltb_shell_results.json') as f:
    res = json.load(f)

lam = np.array(res['lambdas'])
ux_top = np.array(res['ux_tf'])
ux_bot = np.array(res['ux_bf'])
M_applied = lam * M_cr  # loads were scaled to M_cr

# Twist = differential lateral displacement / section depth
twist = np.abs(ux_top - ux_bot) / d

# Perry-Robertson envelope: delta = delta_0 / (1 - M/M_cr)
delta_0 = imp_mag
M_range = np.linspace(0, 0.99 * M_cr, 200)
perry = delta_0 / (1 - M_range / M_cr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# M vs lateral displacement (top flange)
ax1.plot(np.abs(ux_top), M_applied / 1e6, 'b-o', ms=3, label='Shell: top flange')
ax1.plot(np.abs(ux_bot), M_applied / 1e6, 'c-s', ms=2, label='Shell: bot flange')
ax1.plot(perry, M_range / 1e6, 'r--', label='Perry envelope')
ax1.axhline(M_cr / 1e6, color='gray', ls=':', label=f'$M_{{cr}}$ = {M_cr/1e6:.2f} kN·m')
ax1.set_xlabel('Lateral displacement |ux| (mm)')
ax1.set_ylabel('Applied moment M (kN·m)')
ax1.set_title('M vs Lateral Displacement')
ax1.legend()
ax1.set_xlim(left=0)
ax1.grid(True, alpha=0.3)

# M vs twist
ax2.plot(twist * 1000, M_applied / 1e6, 'g-o', ms=3, label='Shell twist')
ax2.axhline(M_cr / 1e6, color='gray', ls=':', label=f'$M_{{cr}}$ = {M_cr/1e6:.2f} kN·m')
ax2.set_xlabel('Twist (ux_top − ux_bot) / d  ×1000')
ax2.set_ylabel('Applied moment M (kN·m)')
ax2.set_title('M vs Twist')
ax2.legend()
ax2.set_xlim(left=0)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Report
print(f"\nClassical M_cr = {M_cr/1e6:.3f} kN·m")
if len(lam) > 1:
    large_idx = np.where(np.abs(ux_top) > 10)[0]
    if len(large_idx) > 0:
        M_buckle = M_applied[large_idx[0]]
        print(f"M at |ux_tf|>10mm = {M_buckle/1e6:.3f} kN·m ({M_buckle/M_cr:.1%} of M_cr)")
    print(f"Max lambda reached: {lam[-1]:.3f} ({lam[-1]*100:.0f}% of M_cr)")

## 9. Capture results — manual + DomainCapture paths

Two ways to produce a native-HDF5 results file consumable by the
apeGmsh ``ResultsViewer``:

1. **Manual path** — query the live OpenSees domain post-analysis,
   open a ``NativeWriter``, and write nodal displacements yourself.
   Good for one-shot snapshots and post-hoc diagnostics.
2. **DomainCapture path** — declare what to capture with
   ``Recorders().nodes(...)``, hand the spec to a ``DomainCapture``
   context, and call ``cap.step(t=...)`` after each ``ops.analyze``
   (the helper does it for you). Scales to multi-stage, transient,
   modal, and multi-recorder runs.

Both produce a file that ``Results.from_native(path).viewer()`` can
open. The viewer launch is gated on ``APEGMSH_SKIP_VIEWER`` so this
notebook is safe to run under nbconvert / CI.


In [ ]:
# --- EOS-WIRING-V1 ---
# Manual path: pull displacements off the live domain, write h5 yourself.
from pathlib import Path
import numpy as np
from apeGmsh.results.writers import NativeWriter

results_manual = Path("example_LTB_shell_manual.h5")
if results_manual.exists():
    results_manual.unlink()

_n = len(fem.nodes.ids)
_ux = np.array([ops.nodeDisp(int(nid), 1) for nid in fem.nodes.ids])
_uy = np.array([ops.nodeDisp(int(nid), 2) for nid in fem.nodes.ids])
_uz = np.array([ops.nodeDisp(int(nid), 3) for nid in fem.nodes.ids])
_components = {
    "displacement_x": _ux.reshape(1, _n),
    "displacement_y": _uy.reshape(1, _n),
    "displacement_z": _uz.reshape(1, _n),
}

with NativeWriter(results_manual) as _nw:
    _nw.open(fem=fem)
    _sid = _nw.begin_stage(name="static", kind="static", time=np.array([1.0]))
    _nw.write_nodes(
        _sid, "partition_0",
        node_ids=np.asarray(fem.nodes.ids, dtype=np.int64),
        components=_components,
    )
    _nw.end_stage()

print(f"manual -> {results_manual} ({results_manual.stat().st_size/1024:.1f} KB)")


In [ ]:
# DomainCapture path: declarative recorders, capture during analyze.
from apeGmsh.solvers.Recorders import Recorders
from apeGmsh.results.capture._domain import DomainCapture

recs = Recorders()
recs.nodes(components="displacement")
recs.nodes(components="reaction_force")
spec = recs.resolve(fem, ndm=3, ndf=6)

results_capture = Path("example_LTB_shell_capture.h5")
if results_capture.exists():
    results_capture.unlink()

with DomainCapture(spec, results_capture, fem, ndm=3, ndf=6) as cap:
    cap.begin_stage("run", kind="static")
    # --- copied from cell 14 ---
    # ── Build the openseespy script ──────────────────────────
    n_steps = 80

    # Node data
    node_lines = []
    for nid, x, y, z in node_data:
        node_lines.append(f"ops.node({nid}, {x:.6f}, {y:.6f}, {z:.6f})")

    # Element lines
    elem_lines = []
    for i in range(len(tf_ids)):
        c = tf_conn[i]
        elem_lines.append(f"ops.element('ASDShellQ4', {int(tf_ids[i])}, {int(c[0])}, {int(c[1])}, {int(c[2])}, {int(c[3])}, 1, '-corotational')")
    for i in range(len(bf_ids)):
        c = bf_conn[i]
        elem_lines.append(f"ops.element('ASDShellQ4', {int(bf_ids[i])}, {int(c[0])}, {int(c[1])}, {int(c[2])}, {int(c[3])}, 1, '-corotational')")
    for i in range(len(web_ids)):
        c = web_conn[i]
        elem_lines.append(f"ops.element('ASDShellQ4', {int(web_ids[i])}, {int(c[0])}, {int(c[1])}, {int(c[2])}, {int(c[3])}, 2, '-corotational')")

    # BCs from face_sp records (direct fix on end nodes — no rigidLink)
    sp_fix = {}  # node_id -> set of dofs
    for rec in fem.nodes.sp:
        sp_fix.setdefault(int(rec.node_id), set()).add(rec.dof)

    fix_lines = []
    for nid, dofs in sp_fix.items():
        mask = [1 if d in dofs else 0 for d in range(1, 7)]
        fix_lines.append(f"ops.fix({nid}, {mask[0]}, {mask[1]}, {mask[2]}, {mask[3]}, {mask[4]}, {mask[5]})")

    # Load lines
    load_lines = []
    for rec in fem.nodes.loads:
        f = rec.force_xyz or (0, 0, 0)
        m = rec.moment_xyz or (0, 0, 0)
        load_lines.append(
            f"ops.load({int(rec.node_id)}, {f[0]:.10g}, {f[1]:.10g}, {f[2]:.10g}, "
            f"{m[0]:.10g}, {m[1]:.10g}, {m[2]:.10g})")

    # Find midspan tracking nodes (top flange and bot flange)
    bf_node_ids = sorted(int(n) for n in fem.nodes.get_ids(label='bot_flange'))
    bf_node_coords = np.array([fem.nodes.coords[id_to_idx[n]] for n in bf_node_ids])
    target_bf = np.array([0, y_bf, L/2])
    midspan_bf = bf_node_ids[np.argmin(np.linalg.norm(bf_node_coords - target_bf, axis=1))]

    print(f"Tracking: TF node {midspan_node}, BF node {midspan_bf}")

    # Assemble script
    script = f"""\
    import openseespy.opensees as ops
    import numpy as np
    import json

    ops.wipe()
    ops.model('basic', '-ndm', 3, '-ndf', 6)

    # ── Nodes ─────────────────────────────────────────────────
    {chr(10).join(node_lines)}

    # ── Sections ──────────────────────────────────────────────
    ops.section('ElasticMembranePlateSection', 1, {E}, {nu}, {tf}, 0.0)   # flanges
    ops.section('ElasticMembranePlateSection', 2, {E}, {nu}, {tw}, 0.0)   # web

    # ── Elements (ASDShellQ4 -corotational) ───────────────────
    {chr(10).join(elem_lines)}

    # ── Boundary conditions (face_sp: direct fix on end nodes) ──
    {chr(10).join(fix_lines)}

    # ── Loading (face_load: distributed moment couples) ───────
    ops.timeSeries('Linear', 1)
    ops.pattern('Plain', 1, 1)
    {chr(10).join(load_lines)}

    # ── Analysis ──────────────────────────────────────────────
    ops.constraints('Plain')
    ops.numberer('Plain')
    ops.system('UmfPack')
    ops.test('NormDispIncr', 1e-5, 100)
    ops.algorithm('Newton')

    n_steps = {n_steps}
    dLambda = 1.5 / n_steps
    ops.integrator('LoadControl', dLambda)
    ops.analysis('Static')

    lambdas = []
    ux_tf = []   # top flange lateral displacement
    ux_bf = []   # bottom flange lateral displacement

    for step in range(n_steps):
        ok = ops.analyze(1)
        cap.step(t=ops.getTime())
        if ok != 0:
            print(f"Analysis failed at step {{step}}, lambda={{ops.getTime():.4f}}")
            break
        lam = ops.getTime()
        u_top = ops.nodeDisp({midspan_node}, 1)
        u_bot = ops.nodeDisp({midspan_bf}, 1)
        lambdas.append(lam)
        ux_tf.append(u_top)
        ux_bf.append(u_bot)
        if step % 10 == 0:
            twist = abs(u_top - u_bot)
            print(f"  step {{step:3d}}: lambda={{lam:.4f}}, ux_tf={{u_top:.3f}}, twist={{twist:.3f}}")

    print(f"\\nCompleted {{len(lambdas)}} steps, final lambda = {{lambdas[-1]:.4f}}")

    results = {{
        'lambdas': lambdas,
        'ux_tf': ux_tf,
        'ux_bf': ux_bf,
        'M_cr': {M_cr},
    }}
    with open('ltb_shell_results.json', 'w', encoding='utf-8') as f:
        json.dump(results, f)
    print("Results saved")
    """

    script_path = 'ltb_shell_analysis.py'
    with open(script_path, 'w', encoding='utf-8') as f:
        f.write(script)
    print(f"Script: {len(script.splitlines())} lines, {len(elem_lines)} elements")
    cap.end_stage()

print(f"capture -> {results_capture} ({results_capture.stat().st_size/1024:.1f} KB)")


In [ ]:
# Open the captured run in the apeGmsh ResultsViewer (subprocess,
# non-blocking). Set APEGMSH_SKIP_VIEWER=1 to skip in headless / CI.
import os
from apeGmsh.results import Results
results = Results.from_native(results_capture)
if os.environ.get("APEGMSH_SKIP_VIEWER"):
    print("[skip viewer] APEGMSH_SKIP_VIEWER set")
else:
    handle = results.viewer(blocking=False)
    print(f"viewer pid: {handle.pid}  -- close window to exit.")


In [ ]:
g.end()
print('Done')